# Zero-shot Classification with Language Models
**Authored by Alexandre Mathias DONNAT, Sr - Télécom Paris**

The goal of this lab are to:
- Familiarize yourself with Large Language Models
    - Understand prompt-based classification with language models
    - Implement classification using next-token logits
- Try to reproduce results described in a scientific paper
- Implement rigorous experiments

We will in this lab aim to produce detailled results following the methodology presented in 
- [Making Pre-trained Language Models Better Few-shot Learners](https://aclanthology.org/2021.acl-long.295.pdf) (Gao et al, 2021)
- [Noisy Channel Language Model Prompting for Few-Shot Text Classification](https://aclanthology.org/2022.acl-long.365.pdf) (Min et al, 2022)


The goal of this paper is to investigate the zero-shot capabilities of relatively small Language Models on classification by **scoring the labels** of the classification tasks, with various *scoring functions*.

### Part I

Assuming a label set of classes $\mathcal{C}$: usually, a sequence of text $x = (w_1, ..., w_l) \in \mathcal{V}^l$ has an associated label $y$, and the classification model learns to predict $$P(y|x)$$

Here, we assume $\mathcal{C}$ to be included in the vocabulary of the model ($\mathcal{C} \cup \mathcal{V})$; and assuming an input context ${x}$, we:
- Create a **prompt** ${x}'$ = $prompt(x)$; the function $prompt$ is task-dependant and given, for example, in Table 1 of the first paper. 
- Instead of generating an answer, we will use **the probability for the first token to be generated** $$P(y|x' = prompt(x)), \forall y \in \mathcal{C}$$
- Use the $\text{Argmax}_{y \in \mathcal{C}} P(y|x')$ as prediction,
    - The first paper uses *label words* as classes. 
- Compute the appropriate metric for the dataset and compare to a simple baseline (*i.e*, random or majority draw).


#### What to do ?

Your job is to implement experiments for:
- Several of the datasets that are **used in both papers**,
- Using a small version of one of the models used (for example ```GPT-2```)
- Use the direct probability computation used in **both papers**, and implement more elaborate scores from the second paper, as presented in Table 1.
Look at the appropriate metrics and compare with the results given in the papers ! 

In [1]:
import os

os.environ["HF_HOME"] = r"C:\hf_cache"
os.environ["HF_DATASETS_CACHE"] = r"C:\hf_cache\datasets"
os.environ["TRANSFORMERS_CACHE"] = r"C:\hf_cache\transformers"

In [ ]:
import torch
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
model_name = "gpt2"
# model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print(f"Model: {model_name}")
print(f"Device: {device}")

In [4]:
text = "The movie was surprisingly touching and well acted."

prompt = f"Review: {text}\nSentiment:"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    logits = model(**inputs).logits[0, -1]

pos_id = tokenizer.encode(" positive", add_special_tokens=False)[0]
neg_id = tokenizer.encode(" negative", add_special_tokens=False)[0]

print("LM-BFF scores:")
print("positive:", logits[pos_id].item())
print("negative:", logits[neg_id].item())

LM-BFF scores:
positive: -141.733642578125
negative: -142.73065185546875


In [5]:
def score(text, label):
    prompt = f"Sentiment: {label}\nReview: {text}"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])

    return -outputs.loss.item()

print("Noisy channel scores:")
print("positive:", score(text, "positive"))
print("negative:", score(text, "negative"))

Noisy channel scores:
positive: -4.980668067932129
negative: -4.976251602172852


In [6]:
dataset = load_dataset(
    "glue",
    "sst2",
    cache_dir=r"C:\hf_cache\datasets"
)

train = dataset["train"]
test = dataset["validation"]

print(dataset)
print(train[0])
print(test[0])

Generating test split: 100%|██████████| 1821/1821 [00:00<00:00, 905384.97 examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}
{'sentence': "it 's a charming and often affecting journey . ", 'label': 1, 'idx': 0}


### Evaluation on SST-2

I first evaluate the zero-shot classification method on SST-2, a binary sentiment classification dataset.

The labels are:

- `0`: negative
- `1`: positive

I compare three methods:

1. A majority-class baseline.
2. A random baseline.
3. Direct prompt-based scoring using GPT-2 next-token logits.
4. Noisy channel scoring using the negative language modeling loss.

For computational reasons, I evaluate the model on a subset of the validation set.

In [21]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report

N_EVAL = 800

eval_data = test.select(range(min(N_EVAL, len(test))))

texts = eval_data["sentence"]
y_true = np.array(eval_data["label"])

print(f"Number of evaluation examples: {len(eval_data)}")
print("Label distribution:")
print(pd.Series(y_true).value_counts().sort_index())

Number of evaluation examples: 800
Label distribution:
0    393
1    407
Name: count, dtype: int64


In [22]:
label_words = {
    0: " negative",
    1: " positive"
}

label_names = {
    0: "negative",
    1: "positive"
}

label_token_ids = {
    label: tokenizer.encode(word, add_special_tokens=False)[0]
    for label, word in label_words.items()
}

print(label_token_ids)

{0: 4633, 1: 3967}


In [23]:
def direct_score(text, label):
    prompt = f"Review: {text}\nSentiment:"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]

    token_id = label_token_ids[label]
    return logits[token_id].item()


def predict_direct(text):
    scores = {
        label: direct_score(text, label)
        for label in label_words
    }

    prediction = max(scores, key=scores.get)
    return prediction, scores

In [25]:
example_text = "The movie was surprisingly touching and well acted."

prediction, scores = predict_direct(example_text)

print("Text:", example_text)
print("Scores:", scores)
print("Prediction:", prediction, "-", label_names[prediction])

Text: The movie was surprisingly touching and well acted.
Scores: {0: -142.73065185546875, 1: -141.733642578125}
Prediction: 1 - positive


In [26]:
def noisy_channel_score(text, label):
    label_word = label_words[label].strip()
    prompt = f"Sentiment: {label_word}\nReview: {text}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])

    return -outputs.loss.item()


def predict_noisy_channel(text):
    scores = {
        label: noisy_channel_score(text, label)
        for label in label_words
    }

    prediction = max(scores, key=scores.get)
    return prediction, scores

In [27]:
prediction, scores = predict_noisy_channel(example_text)

print("Text:", example_text)
print("Scores:", scores)
print("Prediction:", prediction, "-", label_names[prediction])

Text: The movie was surprisingly touching and well acted.
Scores: {0: -4.976251602172852, 1: -4.980668067932129}
Prediction: 0 - negative


In [28]:
def majority_baseline(y_true):
    values, counts = np.unique(y_true, return_counts=True)
    majority_label = values[np.argmax(counts)]
    return np.full_like(y_true, majority_label)


def random_baseline(y_true, seed=42):
    rng = np.random.default_rng(seed)
    labels = np.unique(y_true)
    return rng.choice(labels, size=len(y_true))

In [29]:
y_pred_direct = []

for text in tqdm(texts, desc="Direct label scoring"):
    pred, _ = predict_direct(text)
    y_pred_direct.append(pred)

y_pred_direct = np.array(y_pred_direct)

Direct label scoring: 100%|██████████| 800/800 [01:23<00:00,  9.57it/s]


In [30]:
y_pred_noisy = []

for text in tqdm(texts, desc="Noisy channel scoring"):
    pred, _ = predict_noisy_channel(text)
    y_pred_noisy.append(pred)

y_pred_noisy = np.array(y_pred_noisy)

Noisy channel scoring: 100%|██████████| 800/800 [01:25<00:00,  9.38it/s]


In [31]:
y_pred_majority = majority_baseline(y_true)
y_pred_random = random_baseline(y_true)

results = pd.DataFrame([
    {
        "method": "majority_baseline",
        "accuracy": accuracy_score(y_true, y_pred_majority),
        "macro_f1": f1_score(y_true, y_pred_majority, average="macro")
    },
    {
        "method": "random_baseline",
        "accuracy": accuracy_score(y_true, y_pred_random),
        "macro_f1": f1_score(y_true, y_pred_random, average="macro")
    },
    {
        "method": "direct_label_scoring",
        "accuracy": accuracy_score(y_true, y_pred_direct),
        "macro_f1": f1_score(y_true, y_pred_direct, average="macro")
    },
    {
        "method": "noisy_channel",
        "accuracy": accuracy_score(y_true, y_pred_noisy),
        "macro_f1": f1_score(y_true, y_pred_noisy, average="macro")
    }
])

results.sort_values("accuracy", ascending=False)

,method,accuracy,macro_f1
3,noisy_channel,0.70750,0.700300
2,direct_label_scoring,0.58875,0.510998
0,majority_baseline,0.50875,0.337200
1,random_baseline,0.47375,0.473611


### Analysis of the SST-2 zero-shot results

The results show that prompt-based zero-shot classification with GPT-2 is able to extract some sentiment information from the input text.

On the evaluated subset of SST-2, the majority baseline obtains an accuracy of about 50.9%, while the random baseline obtains about 47.4%. These baselines are close to chance level, which is expected for a balanced binary classification task.

The direct label scoring method improves over both baselines, with an accuracy of about 58.9%. This means that using the next-token logits after the prompt provides some useful signal. However, the improvement remains limited, showing that the method is sensitive to the prompt formulation and to the choice of label words.

The noisy channel method performs clearly better, with an accuracy of about 70.8% and a macro-F1 of about 70.0%. This suggests that, in this setup, scoring the likelihood of the input text conditioned on the candidate label is more robust than directly scoring the next label token.

This difference is consistent with the motivation of noisy channel prompting: direct label scoring can suffer from label-word bias, because the model may prefer some words independently of the input text. The noisy channel formulation partially reduces this issue by evaluating whether the text is more likely under a given label.

These results show that classification can be converted into a language modeling problem by scoring candidate labels. However, the performance depends strongly on the prompt, the verbalizers, the scoring function, and the model size. GPT-2 is not instruction-tuned and remains a relatively small model, so these results should be interpreted as a methodological reproduction rather than a fully optimized classification system.

### Additional experiment: AG News

To make the evaluation more rigorous, I also test the same zero-shot classification strategy on AG News.

AG News is a topic classification dataset with four classes:

- `0`: World
- `1`: Sports
- `2`: Business
- `3`: Science/Technology

This allows me to test whether the prompt-based classification method also works in a multi-class setting, not only on binary sentiment classification.

In [37]:
ag_dataset = load_dataset(
    "ag_news",
    cache_dir=r"C:\hf_cache\datasets"
)

ag_test = ag_dataset["test"]

N_EVAL_AG = 800
ag_eval = ag_test.select(range(min(N_EVAL_AG, len(ag_test))))

ag_texts = ag_eval["text"]
ag_y_true = np.array(ag_eval["label"])

print(ag_dataset)
print(ag_eval[0])
print("Label distribution:")
print(pd.Series(ag_y_true).value_counts().sort_index())

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
{'text': "Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.", 'label': 2}
Label distribution:
0    210
1    213
2    163
3    214
Name: count, dtype: int64


In [38]:
ag_label_words = {
    0: " world",
    1: " sports",
    2: " business",
    3: " technology"
}

ag_label_names = {
    0: "world",
    1: "sports",
    2: "business",
    3: "technology"
}

ag_label_token_ids = {
    label: tokenizer.encode(word, add_special_tokens=False)[0]
    for label, word in ag_label_words.items()
}

print(ag_label_token_ids)

{0: 995, 1: 5701, 2: 1597, 3: 3037}


In [39]:
def ag_direct_score(text, label):
    prompt = f"Article: {text}\nTopic:"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]

    token_id = ag_label_token_ids[label]
    return logits[token_id].item()


def ag_predict_direct(text):
    scores = {
        label: ag_direct_score(text, label)
        for label in ag_label_words
    }

    prediction = max(scores, key=scores.get)
    return prediction, scores

In [40]:
def ag_noisy_channel_score(text, label):
    label_word = ag_label_words[label].strip()
    prompt = f"Topic: {label_word}\nArticle: {text}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])

    return -outputs.loss.item()


def ag_predict_noisy_channel(text):
    scores = {
        label: ag_noisy_channel_score(text, label)
        for label in ag_label_words
    }

    prediction = max(scores, key=scores.get)
    return prediction, scores

In [41]:
ag_y_pred_direct = []

for text in tqdm(ag_texts, desc="AG News — direct label scoring"):
    pred, _ = ag_predict_direct(text)
    ag_y_pred_direct.append(pred)

ag_y_pred_direct = np.array(ag_y_pred_direct)

AG News — direct label scoring: 100%|██████████| 800/800 [04:05<00:00,  3.26it/s]


In [42]:
ag_y_pred_noisy = []

for text in tqdm(ag_texts, desc="AG News — noisy channel scoring"):
    pred, _ = ag_predict_noisy_channel(text)
    ag_y_pred_noisy.append(pred)

ag_y_pred_noisy = np.array(ag_y_pred_noisy)

AG News — noisy channel scoring: 100%|██████████| 800/800 [04:17<00:00,  3.11it/s]


In [43]:
ag_y_pred_majority = majority_baseline(ag_y_true)
ag_y_pred_random = random_baseline(ag_y_true)

ag_results = pd.DataFrame([
    {
        "dataset": "ag_news",
        "method": "majority_baseline",
        "accuracy": accuracy_score(ag_y_true, ag_y_pred_majority),
        "macro_f1": f1_score(ag_y_true, ag_y_pred_majority, average="macro")
    },
    {
        "dataset": "ag_news",
        "method": "random_baseline",
        "accuracy": accuracy_score(ag_y_true, ag_y_pred_random),
        "macro_f1": f1_score(ag_y_true, ag_y_pred_random, average="macro")
    },
    {
        "dataset": "ag_news",
        "method": "direct_label_scoring",
        "accuracy": accuracy_score(ag_y_true, ag_y_pred_direct),
        "macro_f1": f1_score(ag_y_true, ag_y_pred_direct, average="macro")
    },
    {
        "dataset": "ag_news",
        "method": "noisy_channel",
        "accuracy": accuracy_score(ag_y_true, ag_y_pred_noisy),
        "macro_f1": f1_score(ag_y_true, ag_y_pred_noisy, average="macro")
    }
])

ag_results.sort_values("accuracy", ascending=False)

,dataset,method,accuracy,macro_f1
3,ag_news,noisy_channel,0.65000,0.628781
2,ag_news,direct_label_scoring,0.57625,0.557002
0,ag_news,majority_baseline,0.26750,0.105523
1,ag_news,random_baseline,0.25250,0.252269


In [44]:
combined_results = pd.concat(
    [
        results.assign(dataset="sst2"),
        ag_results
    ],
    ignore_index=True
)

combined_results.sort_values(["dataset", "accuracy"], ascending=[True, False])

,method,accuracy,macro_f1,dataset
7,noisy_channel,0.65000,0.628781,ag_news
6,direct_label_scoring,0.57625,0.557002,ag_news
4,majority_baseline,0.26750,0.105523,ag_news
5,random_baseline,0.25250,0.252269,ag_news
3,noisy_channel,0.70750,0.700300,sst2
2,direct_label_scoring,0.58875,0.510998,sst2
0,majority_baseline,0.50875,0.337200,sst2
1,random_baseline,0.47375,0.473611,sst2


### Global analysis of Part I

The results are consistent across both datasets: prompt-based zero-shot classification with GPT-2 performs better than simple baselines, but the performance strongly depends on the scoring function.

On SST-2, the majority baseline reaches about 50.9% accuracy and the random baseline about 47.4%. Direct label scoring improves the result to about 58.9%, while the noisy channel method reaches about 70.8% accuracy. This shows that GPT-2 contains useful sentiment information, but that direct next-token label scoring is still quite fragile.

On AG News, the same pattern appears. The majority and random baselines are close to 25%, which is expected for a four-class classification task. Direct label scoring reaches about 57.6% accuracy, and noisy channel scoring improves the result to about 65.0%. This confirms that the approach is not limited to binary sentiment classification and can also work in a multi-class topic classification setting.

The noisy channel method outperforms direct label scoring on both datasets. This suggests that directly scoring label words can suffer from label-word bias: GPT-2 may prefer some tokens independently of the input text. By contrast, the noisy channel formulation evaluates how likely the text is when conditioned on a candidate label, which seems more robust in these experiments.

However, the results remain far from a fully supervised classifier. This is expected because GPT-2 is a relatively small causal language model and is not instruction-tuned. The experiment should therefore be interpreted as a reproduction of the prompting methodology rather than as an optimized classification system.

Part I shows that text classification can be converted into a language modeling problem by scoring candidate labels. The main experimental lesson is that performance depends heavily on the prompt, the label words, the scoring function, and the model.

### Few-shot variants: concat-based and ensemble-based demonstrations

In addition to pure zero-shot prompting, I also test two few-shot prompting variants inspired by the noisy channel paper:

1. **Concat-based demonstrations**: several labeled examples are concatenated before the test example in a single prompt.
2. **Ensemble-based demonstrations**: each labeled example is used separately as a demonstration, and the final score is obtained by averaging the scores across demonstrations.

The goal is to test whether adding a few labeled examples in the prompt improves classification compared with the pure zero-shot setting.

In [59]:
# We use a small balanced set of demonstrations from the SST-2 training set.
# These examples are only used inside the prompt, not for gradient-based training.

N_DEMOS_PER_CLASS = 2

demo_indices = []

for label in [0, 1]:
    label_indices = [i for i, y in enumerate(train["label"]) if y == label]
    demo_indices.extend(label_indices[:N_DEMOS_PER_CLASS])

demos = train.select(demo_indices)

print(f"Number of demonstrations: {len(demos)}")
for ex in demos:
    print(ex["label"], "-", ex["sentence"])

Number of demonstrations: 4
0 - hide new secretions from the parental units 
0 - contains no wit , only labored gags 
1 - that loves its characters and communicates something rather beautiful about human nature 
1 - demonstrates that the director of such hollywood blockbusters as patriot games can still turn out a small , personal film with an emotional wallop . 


In [60]:
def format_demo(example):
    label = "positive" if example["label"] == 1 else "negative"
    return f"Review: {example['sentence']}\nSentiment: {label}\n"


def format_all_demos(demos):
    return "\n".join(format_demo(example) for example in demos)

In [61]:
def format_demo(example):
    label = "positive" if example["label"] == 1 else "negative"
    return f"Review: {example['sentence']}\nSentiment: {label}\n"


def format_all_demos(demos):
    return "\n".join(format_demo(example) for example in demos)

In [62]:
def concat_direct_score(text, label, demos):
    demonstrations = format_all_demos(demos)

    prompt = (
        demonstrations
        + "\n"
        + f"Review: {text}\nSentiment:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]

    token_id = label_token_ids[label]
    return logits[token_id].item()


def predict_concat_direct(text, demos):
    scores = {
        label: concat_direct_score(text, label, demos)
        for label in label_words
    }

    prediction = max(scores, key=scores.get)
    return prediction, scores

In [63]:
def ensemble_direct_score(text, label, demos):
    scores = []

    for demo in demos:
        demo_text = format_demo(demo)

        prompt = (
            demo_text
            + "\n"
            + f"Review: {text}\nSentiment:"
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits[0, -1]

        token_id = label_token_ids[label]
        scores.append(logits[token_id].item())

    return np.mean(scores)


def predict_ensemble_direct(text, demos):
    scores = {
        label: ensemble_direct_score(text, label, demos)
        for label in label_words
    }

    prediction = max(scores, key=scores.get)
    return prediction, scores

In [64]:
y_pred_concat = []

for text in tqdm(texts, desc="Concat-based direct scoring"):
    pred, _ = predict_concat_direct(text, demos)
    y_pred_concat.append(pred)

y_pred_concat = np.array(y_pred_concat)

Concat-based direct scoring: 100%|██████████| 800/800 [04:06<00:00,  3.25it/s]


In [65]:
y_pred_ensemble = []

for text in tqdm(texts, desc="Ensemble-based direct scoring"):
    pred, _ = predict_ensemble_direct(text, demos)
    y_pred_ensemble.append(pred)

y_pred_ensemble = np.array(y_pred_ensemble)

Ensemble-based direct scoring: 100%|██████████| 800/800 [08:32<00:00,  1.56it/s]


In [66]:
fewshot_results = pd.DataFrame([
    {
        "dataset": "sst2",
        "method": "concat_based_direct",
        "accuracy": accuracy_score(y_true, y_pred_concat),
        "macro_f1": f1_score(y_true, y_pred_concat, average="macro")
    },
    {
        "dataset": "sst2",
        "method": "ensemble_based_direct",
        "accuracy": accuracy_score(y_true, y_pred_ensemble),
        "macro_f1": f1_score(y_true, y_pred_ensemble, average="macro")
    }
])

fewshot_results.sort_values("accuracy", ascending=False)

,dataset,method,accuracy,macro_f1
0,sst2,concat_based_direct,0.71125,0.699062
1,sst2,ensemble_based_direct,0.54125,0.406676


In [67]:
sst2_part1_extended = pd.concat(
    [
        results.assign(dataset="sst2"),
        fewshot_results
    ],
    ignore_index=True
)

sst2_part1_extended.sort_values("accuracy", ascending=False)

,method,accuracy,macro_f1,dataset
4,concat_based_direct,0.71125,0.699062,sst2
3,noisy_channel,0.70750,0.700300,sst2
2,direct_label_scoring,0.58875,0.510998,sst2
5,ensemble_based_direct,0.54125,0.406676,sst2
0,majority_baseline,0.50875,0.337200,sst2
1,random_baseline,0.47375,0.473611,sst2


### Analysis of few-shot prompting variants

The concat-based and ensemble-based variants extend the zero-shot setup by adding labeled demonstrations to the prompt.

In the concat-based method, all demonstrations are inserted before the test example in a single prompt. This gives the model an explicit pattern to imitate: each review is followed by its sentiment label. In my experiment, concat-based direct scoring reaches about 71.1% accuracy and 69.9% macro-F1 on SST-2. This is slightly better than the zero-shot noisy channel method in terms of accuracy, and clearly better than zero-shot direct label scoring.

In the ensemble-based method, each demonstration is used separately, and the final score is averaged across demonstrations. In my experiment, ensemble-based direct scoring reaches only about 54.1% accuracy and 40.7% macro-F1. This result is weaker than concat-based prompting and even weaker than zero-shot direct scoring. This suggests that, in this setup, averaging scores across single-demonstration prompts does not provide a robust signal.

These few-shot variants are conceptually different from the fine-tuning procedure in Part II. Here, the model parameters are not updated. The labeled examples are only provided in the input context. Therefore, this remains an in-context learning experiment rather than gradient-based training.

Concat-based prompting appears useful in this experiment, while ensemble-based prompting is less effective. This confirms that few-shot prompting is highly sensitive to the prompt structure, the selected demonstrations, and the aggregation method.

### Part II

Picking one of the dataset you experimented with in Part I, fine-tune your model in the same way as described in **Section 4.1** of the first paper. Report the results and compare with those you obtained earlier.
*Note that the goal of this part is to make you understand and implement an unusual training procedure - and not to obtain better results. Negative results could be expected and are not a bad things, as long as we analyze them.*

### Prompt-based fine-tuning on SST-2

For Part II, I use SST-2 because it is the simplest dataset among the experiments from Part I. It is a binary sentiment classification task with two label words: `negative` and `positive`.

The goal is not to train a standard classifier with a classification head. Instead, I keep the language modeling formulation: the model receives a prompt and is trained to generate the correct label word.

For an input sentence, the training sequence is:

`Review: sentence  
Sentiment: label`

The loss is computed only on the label tokens, while the prompt tokens are masked. This means that the model is optimized to increase the probability of the correct label word after the prompt.

In [45]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

In [46]:
# Few-shot setting: small number of examples per class
K = 32

few_shot_indices = []

for label in [0, 1]:
    label_indices = [i for i, y in enumerate(train["label"]) if y == label]
    few_shot_indices.extend(label_indices[:K])

few_shot_train = train.select(few_shot_indices)

print(f"Few-shot training size: {len(few_shot_train)}")
print(pd.Series(few_shot_train["label"]).value_counts().sort_index())
print(few_shot_train[0])

Few-shot training size: 64
0    32
1    32
Name: count, dtype: int64
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


In [49]:
class PromptClassificationDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_length=128):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.label_words = {
            0: " negative",
            1: " positive"
        }

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        text = self.dataset[idx]["sentence"]
        label = self.dataset[idx]["label"]

        prompt_without_label = f"Review: {text}\nSentiment:"
        label_word = self.label_words[label]

        full_text = prompt_without_label + label_word

        encoded_full = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        encoded_prompt = self.tokenizer(
            prompt_without_label,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        input_ids = encoded_full["input_ids"].squeeze(0)
        attention_mask = encoded_full["attention_mask"].squeeze(0)

        labels = input_ids.clone()

        # Mask prompt tokens: the loss is computed only on the label part
        prompt_length = encoded_prompt["input_ids"].shape[1]
        labels[:prompt_length] = -100

        # Mask padding tokens
        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [50]:
train_dataset_ft = PromptClassificationDataset(
    few_shot_train,
    tokenizer,
    max_length=128
)

train_loader = DataLoader(
    train_dataset_ft,
    batch_size=4,
    shuffle=True
)

In [51]:
ft_model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ft_model.resize_token_embeddings(len(tokenizer))
ft_model.to(device)
ft_model.train()

optimizer = AdamW(ft_model.parameters(), lr=5e-5)

In [52]:
num_epochs = 3

training_losses = []

for epoch in range(num_epochs):
    epoch_losses = []

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}")

    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = ft_model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())
        progress_bar.set_postfix({"loss": np.mean(epoch_losses)})

    avg_loss = np.mean(epoch_losses)
    training_losses.append(avg_loss)

    print(f"Epoch {epoch + 1} — average loss: {avg_loss:.4f}")

Epoch 1/3: 100%|██████████| 16/16 [00:30<00:00,  1.89s/it, loss=1.75]


Epoch 1 — average loss: 1.7453


Epoch 2/3: 100%|██████████| 16/16 [00:29<00:00,  1.86s/it, loss=0.744]


Epoch 2 — average loss: 0.7443


Epoch 3/3: 100%|██████████| 16/16 [00:27<00:00,  1.74s/it, loss=0.531]

Epoch 3 — average loss: 0.5313


In [53]:
def predict_direct_with_model(text, model_to_use):
    prompt = f"Review: {text}\nSentiment:"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        logits = model_to_use(**inputs).logits[0, -1]

    scores = {}

    for label, token_id in label_token_ids.items():
        scores[label] = logits[token_id].item()

    prediction = max(scores, key=scores.get)

    return prediction, scores

In [54]:
ft_model.eval()

y_pred_ft = []

for text in tqdm(texts, desc="Fine-tuned direct scoring on SST-2"):
    pred, _ = predict_direct_with_model(text, ft_model)
    y_pred_ft.append(pred)

y_pred_ft = np.array(y_pred_ft)

ft_results = pd.DataFrame([
    {
        "method": "fine_tuned_prompt_scoring",
        "accuracy": accuracy_score(y_true, y_pred_ft),
        "macro_f1": f1_score(y_true, y_pred_ft, average="macro"),
        "dataset": "sst2"
    }
])

ft_results

Fine-tuned direct scoring on SST-2: 100%|██████████| 800/800 [00:48<00:00, 16.58it/s]


,method,accuracy,macro_f1,dataset
0,fine_tuned_prompt_scoring,0.79,0.789999,sst2


In [55]:
sst2_comparison = pd.concat(
    [
        results.assign(dataset="sst2"),
        ft_results
    ],
    ignore_index=True
)

sst2_comparison.sort_values("accuracy", ascending=False)

,method,accuracy,macro_f1,dataset
4,fine_tuned_prompt_scoring,0.79000,0.789999,sst2
3,noisy_channel,0.70750,0.700300,sst2
2,direct_label_scoring,0.58875,0.510998,sst2
0,majority_baseline,0.50875,0.337200,sst2
1,random_baseline,0.47375,0.473611,sst2


In [56]:
print(classification_report(
    y_true,
    y_pred_ft,
    target_names=["negative", "positive"]
))

              precision    recall  f1-score   support

    negative       0.78      0.80      0.79       393
    positive       0.80      0.78      0.79       407

    accuracy                           0.79       800
   macro avg       0.79      0.79      0.79       800
weighted avg       0.79      0.79      0.79       800



### Analysis of Part II fine-tuning results

The prompt-based fine-tuning procedure improves the performance on SST-2.

Before fine-tuning, the best zero-shot method was the noisy channel scoring method, with an accuracy of about 70.8%. Direct label scoring reached only about 58.9%. After prompt-based fine-tuning, the direct scoring method reaches about 79.0% accuracy and a macro-F1 score of about 79.0%.

This improvement shows that the model has learned to better associate the prompt format with the correct sentiment label. Importantly, the model is still not trained with a standard classification head. It remains a causal language model, and the prediction is still obtained by scoring the candidate label words after the prompt.

The classification report shows that the model is balanced across both classes. The negative class obtains around 78% precision and 80% recall, while the positive class obtains around 80% precision and 78% recall. This suggests that the fine-tuned model does not simply collapse into predicting one class more often than the other.

Compared with the zero-shot setting, fine-tuning reduces the instability of direct label scoring. In the zero-shot case, GPT-2 may prefer some label tokens independently of the input. After fine-tuning, the model becomes more adapted to the specific prompt structure:

`Review: sentence`
`Sentiment: label`

The experiment confirms the main idea of prompt-based fine-tuning: a language model can be adapted to a classification task without adding a classical classification head. The task remains formulated as language modeling, but the model becomes better at assigning high probability to the correct label word.

## General conclusion

In this lab, I implemented prompt-based classification with GPT-2 by treating text classification as a language modeling problem.

The main idea was not to train a classical classifier that directly predicts a class. Instead, I used the language model to score candidate label words and selected the label with the highest score.

In Part I, I first evaluated two zero-shot scoring methods on SST-2 and AG News. Direct label scoring performed better than simple baselines, but the noisy channel method performed better on both datasets. On SST-2, noisy channel scoring reached about 70.8% accuracy, and on AG News it reached about 65.0% accuracy. This suggests that noisy channel scoring can reduce some label-word bias compared with direct next-token scoring.

I also experimented with few-shot prompting variants on SST-2. In the concat-based method, a few labeled examples were concatenated before the test example in a single prompt. This reached about 71.1% accuracy, slightly above the zero-shot noisy channel method. In the ensemble-based method, each demonstration was used separately and the scores were averaged. This performed worse, with about 54.1% accuracy. These results show that adding demonstrations can help, but only if the prompt structure and aggregation method are effective.

In Part II, I fine-tuned GPT-2 on a small few-shot subset of SST-2 using a prompt-based language modeling objective. The model was trained to generate the correct label word after the prompt, while the loss on the prompt tokens was masked. After fine-tuning, the model reached about 79.0% accuracy on SST-2, improving over both zero-shot scoring and few-shot prompting without parameter updates.

The main lesson is that prompt-based classification is powerful but sensitive. The results depend strongly on the prompt template, the verbalizers, the scoring function, the model size, the use of demonstrations, and the fine-tuning procedure. Simple baselines are therefore essential to interpret the results correctly.